In [ ]:
!nvidia-smi
!pip install openai-whisper pydub torch torchaudio -q
!apt install -y ffmpeg -qq
print("✅ Instalación completada")

In [ ]:
import whisper
import torch
import torchaudio
import numpy as np
import pandas as pd
import os, glob, json, zipfile, time
from datetime import datetime
from pydub import AudioSegment
from pydub.effects import normalize
from google.colab import files

print(f"CUDA disponible: {torch.cuda.is_available()}")


In [ ]:
CONFIG = {
    "model_size": "large",
    "language": "es",

  

    "no_speech_threshold": 0.90,        # era 0.45 
    "compression_ratio_threshold": 3.5,
    "logprob_threshold": -3,         

    "temperature": [0.0, 0.2, 0.4, 0.6, 0.8],
    "condition_on_previous_text": True, 
                                        #   ayudar a transcribir voz baja con contexto
    "initial_prompt": (
        "Transcripción de llamada telefónica en español. "
        "Operador de hotel contesta al cliente. "
        "Incluye todo el diálogo aunque haya ruido de fondo o volumen bajo."
    ),

    # VAD 
    "vad_threshold": 0.15,             
    "vad_min_silence_ms": 250,
    "keep_silence_ms": 900,
    "max_retries": 2,
}


def transcribe_with_retries(audio_path: str, audio_array: np.ndarray) -> tuple[str, dict, str]:
    """
    Ahora pasa explícitamente los 3 parámetros de filtrado interno
    que Whisper usa para decidir si descarta o no cada ventana de 30s.
    """
    result = None

    for attempt in range(1, CONFIG["max_retries"] + 2):
        try:
            print(f"    🔄 Intento {attempt}...")
            result = model.transcribe(
                audio_array,
                language=CONFIG["language"],

                # ── Los 3 filtros internos ahora bajo control ──
                no_speech_threshold=CONFIG["no_speech_threshold"],
                compression_ratio_threshold=CONFIG["compression_ratio_threshold"],
                logprob_threshold=CONFIG["logprob_threshold"],

                temperature=CONFIG["temperature"],
                condition_on_previous_text=CONFIG["condition_on_previous_text"],
                initial_prompt=CONFIG["initial_prompt"],

                # ── Ayuda a diagnosticar qué descarta Whisper ──
                
                verbose=False,
            )
            text = result["text"].strip()

            if text:
                print(f"    ✅ OK — {len(text)} chars | "
                      f"Segmentos: {len(result.get('segments', []))}")
                return text, result, "ok"
            else:
                print(f"    ⚠️ Vacío en intento {attempt}")

        except Exception as e:
            print(f"    ❌ Error intento {attempt}: {e}")

    # ── Fallback: sin VAD + parámetros máximamente permisivos ─
    print("    🆘 Fallback: sin preprocesamiento, parámetros permisivos...")
    try:
        result_raw = model.transcribe(
            audio_path,                        # audio original sin tocar
            language=CONFIG["language"],
            no_speech_threshold=0.95,          # casi nunca descarta
            compression_ratio_threshold=4.0,   # acepta casi todo
            logprob_threshold=-3.0,            # acepta confianza muy baja
            temperature=[0.0, 0.2, 0.4, 0.6],
            condition_on_previous_text=True,
            initial_prompt=CONFIG["initial_prompt"],
            verbose=False,
        )
        text_raw = result_raw["text"].strip()
        if text_raw:
            print(f"    ✅ Fallback exitoso — {len(text_raw)} chars")
            return text_raw, result_raw, "ok_fallback"
    except Exception as e:
        print(f"    ❌ Fallback también falló: {e}")

    return "", result, "vacio"

In [ ]:
print(f"Cargando modelo Whisper '{CONFIG['model_size']}'...")
model = whisper.load_model(CONFIG["model_size"])
print("✅ Modelo Whisper cargado")

In [ ]:
print("Cargando modelo Silero VAD...")
vad_model, vad_utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    force_reload=False,
    trust_repo=True,
)
(get_speech_timestamps, _, read_audio, *_) = vad_utils
print("✅ Silero VAD cargado")



In [ ]:
def extract_voice_with_vad(audio_path: str) -> tuple[np.ndarray, bool]:
    """
    Pipeline completo:
    1. Cargar y normalizar
    2. Silero VAD detecta SOLO segmentos con voz humana
       (ignora música de espera y ruido de línea aunque sean ruidosos)
    3. Concatenar solo los segmentos de voz
    4. Retornar array float32 para Whisper

    Maneja: pregrabado → música/silencio → conversación ✅
    Maneja: voz baja con ruido de fondo ✅
    Maneja: ruido de línea telefónica ✅
    """
    try:
        # ── 1. Cargar con pydub para normalizar ──────────────
        audio_pydub = AudioSegment.from_file(audio_path)
        audio_pydub = audio_pydub.set_channels(1).set_frame_rate(16000)
        audio_pydub = normalize(audio_pydub)

        print(f"    📊 Original: {len(audio_pydub)/1000:.1f}s | dBFS: {audio_pydub.dBFS:.1f}")

        # ── 2. Convertir a tensor para Silero VAD ─────────────
        # Silero VAD requiere float32 en rango [-1, 1] a 16kHz
        samples = np.array(audio_pydub.get_array_of_samples(), dtype=np.float32)
        samples /= np.iinfo(audio_pydub.array_type).max
        wav_tensor = torch.from_numpy(samples)

        # ── 3. Detectar segmentos de VOZ con Silero VAD ───────
        speech_timestamps = get_speech_timestamps(
            wav_tensor,
            vad_model,
            threshold=CONFIG["vad_threshold"],
            min_silence_duration_ms=CONFIG["vad_min_silence_ms"],
            min_speech_duration_ms=50,   # ignora clicks/artefactos < 100ms
            return_seconds=False,          # retorna en muestras (samples)
        )

        if not speech_timestamps:
            print("    ⚠️ VAD no detectó voz — usando audio original como fallback")
            return samples, True  # deja que Whisper lo intente igual

        # ── 4. Extraer y concatenar solo los fragmentos de voz ─
        sr = 16000
        buffer_samples = int(CONFIG["keep_silence_ms"] / 1000 * sr)
        total_original = len(samples)
        voice_chunks = []
        total_voice_ms = 0
        total_skipped_ms = 0

        for seg in speech_timestamps:
            start = max(0, seg["start"] - buffer_samples)
            end   = min(total_original, seg["end"] + buffer_samples)
            voice_chunks.append(samples[start:end])
            total_voice_ms += (end - start) / sr * 1000

        # Calcular cuánto se descartó (música + silencios + ruido)
        total_skipped_ms = len(samples) / sr * 1000 - total_voice_ms

        voice_audio = np.concatenate(voice_chunks)

        print(f"    🎙️ Segmentos de voz: {len(speech_timestamps)} | "
              f"Voz: {total_voice_ms/1000:.1f}s | "
              f"Descartado (música/ruido/silencio): {total_skipped_ms/1000:.1f}s")

        # ── 5. Verificar que quede audio procesable ───────────
        if len(voice_audio) < 8000:  # < 0.5s a 16kHz
            print("    ⛔ Muy poco audio de voz detectado")
            return None, False

        return voice_audio, True

    except Exception as e:
        print(f"    ❌ Preprocesamiento falló: {e}")
        return None, False


In [ ]:
# TRANSCRIPCIÓN (sin cambios relevantes)
# ============================================================
def transcribe_with_retries(audio_path: str, audio_array: np.ndarray) -> tuple[str, dict, str]:
    result = None

    for attempt in range(1, CONFIG["max_retries"] + 2):
        try:
            print(f"    🔄 Intento {attempt}...")
            result = model.transcribe(
                audio_array,
                language=CONFIG["language"],
                no_speech_threshold=CONFIG["no_speech_threshold"],
                temperature=CONFIG["temperature"],
                condition_on_previous_text=CONFIG["condition_on_previous_text"],
                initial_prompt=CONFIG["initial_prompt"],
                verbose=False,
            )
            text = result["text"].strip()

            if text:
                print(f"    ✅ OK — {len(text)} chars | "
                      f"Segmentos: {len(result.get('segments', []))}")
                return text, result, "ok"
            else:
                print(f"    ⚠️ Vacío en intento {attempt}")

        except Exception as e:
            print(f"    ❌ Error intento {attempt}: {e}")

    # ── Fallback: audio crudo sin preprocesar ─────────────────
    # Por si el VAD fue demasiado agresivo y cortó voz real
    print("    🆘 Fallback: transcribiendo audio original sin VAD...")
    try:
        result_raw = model.transcribe(
            audio_path,
            language=CONFIG["language"],
            no_speech_threshold=0.1,
            temperature=[0.0, 0.2, 0.4, 0.6, 0.8],
            condition_on_previous_text=False,
            initial_prompt=CONFIG["initial_prompt"],
            verbose=False,
        )
        text_raw = result_raw["text"].strip()
        if text_raw:
            print(f"    ✅ Fallback exitoso — {len(text_raw)} chars")
            return text_raw, result_raw, "ok_fallback"
    except Exception as e:
        print(f"    ❌ Fallback también falló: {e}")

    return "", result, "vacio"



In [8]:
# DIAGNÓSTICO
# ============================================================
def analyze_segments(result: dict) -> dict:
    segments = result.get("segments", []) if result else []
    if not segments:
        return {"num_segments": 0, "avg_no_speech_prob": None}
    probs = [s.get("no_speech_prob", 0) for s in segments]
    return {
        "num_segments": len(segments),
        "avg_no_speech_prob": round(np.mean(probs), 3),
        "max_no_speech_prob": round(max(probs), 3),
    }


In [9]:
# PIPELINE PRINCIPAL
# ============================================================
def transcribe_audio_files(audio_folder: str, output_file: str = "transcripciones.csv",
                            already_processed: set = None):
    already_processed = already_processed or set()

    exts = ['*.wav', '*.mp3', '*.m4a', '*.flac', '*.aac', '*.ogg']
    audio_files = []
    for ext in exts:
        audio_files.extend(glob.glob(os.path.join(audio_folder, ext)))
        audio_files.extend(glob.glob(os.path.join(audio_folder, "**", ext), recursive=True))

    audio_files = list({f for f in audio_files
                        if os.path.basename(f) not in already_processed})
    print(f"📁 Archivos a procesar: {len(audio_files)}")

    results, failed = [], []
    start_time = time.time()

    for i, audio_file in enumerate(audio_files):
        filename = os.path.basename(audio_file)
        print(f"\n[{i+1}/{len(audio_files)}] 📝 {filename}")

        # ← CAMBIO: ahora usa VAD en lugar de detección por amplitud
        audio_array, ok = extract_voice_with_vad(audio_file)
        if not ok:
            failed.append({"archivo": filename, "motivo": "preprocesamiento_fallido"})
            continue

        text, result, status = transcribe_with_retries(audio_file, audio_array)
        seg_info = analyze_segments(result)

        if not text:
            failed.append({"archivo": filename, "motivo": "transcripcion_vacia", **seg_info})

        results.append({
            "archivo": filename,
            "transcripcion": text,
            "idioma": result.get("language", "error") if result else "error",
            "status": status,
            **seg_info,
            "procesado_en": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        })

        if (i + 1) % 5 == 0:
            _save(results, output_file, failed, final=False)

    df = _save(results, output_file, failed, final=True)
    _summary(results, failed, start_time)
    return df


def _save(results, output_file, failed, final=False):
    if not results:
        return None
    prefix = "" if final else "progreso_"
    df = pd.DataFrame(results)
    df.to_csv(f"{prefix}{output_file}", index=False, encoding="utf-8-sig")
    json_out = f"{prefix}{output_file.replace('.csv', '.json')}"
    with open(json_out, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    if failed:
        pd.DataFrame(failed).to_csv(f"fallos_{output_file}", index=False)
    return df


def _summary(results, failed, start_time):
    total = time.time() - start_time
    ok = sum(1 for r in results if "ok" in r["status"])
    print(f"\n{'='*45}")
    print(f"  Procesados : {len(results)}")
    print(f"  Exitosos   : {ok}")
    print(f"  Fallidos   : {len(failed)}")
    print(f"  Tiempo     : {total/60:.1f} min")
    print(f"{'='*45}")


In [ ]:
# EJECUCIÓN
# ============================================================
uploaded = files.upload()
AUDIO_FOLDER = "/content/audios"
os.makedirs(AUDIO_FOLDER, exist_ok=True)

for filename in uploaded.keys():
    if filename.lower().endswith(".zip"):
        with zipfile.ZipFile(filename, 'r') as z:
            z.extractall(AUDIO_FOLDER)
    else:
        os.rename(filename, os.path.join(AUDIO_FOLDER, filename))

df_results = transcribe_audio_files(AUDIO_FOLDER, "mis_transcripciones.csv")

if df_results is not None:
    print(df_results[["archivo", "status", "avg_no_speech_prob", "transcripcion"]].head(10))
    files.download("mis_transcripciones.csv")
    files.download("mis_transcripciones.json")